# Procesamiento de Lenguaje Natural

**Objetivo:**
Guía que implementa partes típicas de una cadena de Retrieval-Augmented Generation (RAG)


In [1]:
%pip -q install -U \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-ollama \
    sentence-transformers \
    chromadb \
    requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [1]:
# Después de instalar las dependencias, reinicie el entorno una sola vez
# si Colab muestra un mensaje solicitándolo.
print("Dependencias instaladas.")


Dependencias instaladas.


In [2]:
# Setup general
import os
import time
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

print("Entorno configurado correctamente.")


Entorno configurado correctamente.


In [3]:
def crear_sesion_http():
    """Crea una sesión HTTP con reintentos y un User-Agent válido."""
    session = requests.Session()
    retries = Retry(
        total=4,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.headers.update({
        "User-Agent": (
            "PUCP-NLP-RAG-Notebook/1.0 "
            "(educational use; contact: classroom@example.com)"
        )
    })
    return session


def cargar_wikipedia(consulta, max_documentos=3, idioma="en"):
    """Busca artículos y descarga su texto mediante la API oficial de MediaWiki.

    Esta implementación evita el JSONDecodeError que puede producir
    WikipediaLoader cuando Wikipedia devuelve temporalmente HTML o una
    respuesta vacía.
    """
    session = crear_sesion_http()
    api_url = f"https://{idioma}.wikipedia.org/w/api.php"

    search_response = session.get(
        api_url,
        params={
            "action": "query",
            "list": "search",
            "srsearch": consulta,
            "srlimit": max_documentos,
            "format": "json",
            "formatversion": 2,
        },
        timeout=30,
    )
    search_response.raise_for_status()

    content_type = search_response.headers.get("content-type", "")
    if "json" not in content_type.lower():
        raise RuntimeError(
            "Wikipedia no devolvió JSON. "
            f"Tipo de contenido recibido: {content_type or 'desconocido'}"
        )

    search_data = search_response.json()
    resultados = search_data.get("query", {}).get("search", [])

    if not resultados:
        raise ValueError(f"No se encontraron artículos para: {consulta!r}")

    titulos = [resultado["title"] for resultado in resultados]

    extract_response = session.get(
        api_url,
        params={
            "action": "query",
            "prop": "extracts|info",
            "explaintext": 1,
            "inprop": "url",
            "redirects": 1,
            "titles": "|".join(titulos),
            "format": "json",
            "formatversion": 2,
        },
        timeout=30,
    )
    extract_response.raise_for_status()

    content_type = extract_response.headers.get("content-type", "")
    if "json" not in content_type.lower():
        raise RuntimeError(
            "Wikipedia no devolvió JSON al descargar los artículos. "
            f"Tipo recibido: {content_type or 'desconocido'}"
        )

    pages = extract_response.json().get("query", {}).get("pages", [])
    documents = []

    for page in pages:
        texto = page.get("extract", "").strip()
        if not texto:
            continue

        documents.append(
            Document(
                page_content=texto,
                metadata={
                    "title": page.get("title", ""),
                    "source": page.get("fullurl", ""),
                    "pageid": page.get("pageid"),
                },
            )
        )

    if not documents:
        raise RuntimeError("Wikipedia encontró resultados, pero no devolvió texto utilizable.")

    return documents


## Parte 1: Procesamiento de Documentos
Cambie la palabra COMPLETAR por una cadena con un tema cualquiera de su interés (en inglés) y ejecute las celdas. Se cargarán 3 documentos de Wikipedia relacionados a este tema

In [4]:
# Cargar páginas de Wikipedia
tema = "Stephen Wolfram"

documents = cargar_wikipedia(
    consulta=tema,
    max_documentos=3,
    idioma="en",
)

print(f"Loaded {len(documents)} documents with titles:")
for document in documents:
    print("-", document.metadata["title"], document.metadata["source"])

print("\nSample content:")
print(documents[0].page_content[:300] + "...")


Loaded 1 documents with titles:
- Stephen Wolfram https://en.wikipedia.org/wiki/Stephen_Wolfram

Sample content:
Stephen Wolfram ( WUUL-frəm; born 29 August 1959) is a British-American computer scientist, physicist, and businessman. He works in computer algebra and theoretical physics. In 2012, he was named a fellow of the American Mathematical Society. 
As a businessman, Wolfram is the founder and CEO of the ...


In [5]:
# Comparación de estrategias de división de texto
print("--- Using CharacterTextSplitter ---")

char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)

char_splits = char_splitter.split_documents(documents)
max_chunks = 5

print(f"Split into {len(char_splits)} chunks")
for indice, fragmento in enumerate(char_splits[:max_chunks]):
    print(f"\nChunk {indice}:")
    print(fragmento.page_content)

print("\n--- Using RecursiveCharacterTextSplitter ---")

rec_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)

rec_splits = rec_splitter.split_documents(documents)

print(f"Split into {len(rec_splits)} chunks")
for indice, fragmento in enumerate(rec_splits[:max_chunks]):
    print(f"\nChunk {indice}:")
    print(fragmento.page_content)


--- Using CharacterTextSplitter ---
Split into 26 chunks

Chunk 0:
Stephen Wolfram ( WUUL-frəm; born 29 August 1959) is a British-American computer scientist, physicist, and businessman. He works in computer algebra and theoretical physics. In 2012, he was named a fellow of the American Mathematical Society. 
As a businessman, Wolfram is the founder and CEO of the software company Wolfram Research, where he works as chief designer of Mathematica and the Wolfram Alpha answer engine.
== Early life ==
=== Family ===

Chunk 1:
== Early life ==
=== Family ===
Stephen Wolfram was born in London in 1959 to Hugo and Sybil Wolfram, both German Jewish refugees to the United Kingdom. His maternal grandmother was British psychoanalyst Kate Friedlander.

Chunk 2:
Wolfram's father, Hugo Wolfram, was a textile manufacturer and served as managing director of the Lurex Company—makers of the fabric Lurex. Wolfram's mother, Sybil Wolfram, was a Fellow and Tutor in Philosophy at Lady Margaret Hall, a cons

## Parte 2: Vector Store y Recuperación
En esta sección se convertirán los chunks de texto en embeddings usando dos modelos distinto. Luego, se almacenarán los vectores en ChromaDB, una base de datos de vectores que permite almacenamiento local de forma rápida. Finalmente, se probará el sistema con consultas reales y se analizarán los resultados

In [6]:
from pathlib import Path
import shutil

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Modelo 1: rápido y ligero
embeddings_model_1 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Modelo 2: más grande y normalmente más preciso
embeddings_model_2 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Limpiar bases anteriores para que la celda se pueda volver a ejecutar
for directory in ("./small_db", "./large_db"):
    path = Path(directory)
    if path.exists():
        shutil.rmtree(path)

vectorstore_1 = Chroma.from_documents(
    documents=rec_splits,
    embedding=embeddings_model_1,
    persist_directory="./small_db",
    collection_name="wikipedia_minilm",
)

vectorstore_2 = Chroma.from_documents(
    documents=rec_splits,
    embedding=embeddings_model_2,
    persist_directory="./large_db",
    collection_name="wikipedia_mpnet",
)

print("Vector stores creados correctamente.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector stores creados correctamente.


In [7]:
import time

query = "What is the most recent research by Stephen Wolfram?"  # Ingresa una pregunta (en inglés) de acuerdo al tema que hayas colocado arriba

# Resultados modelo 1
print("Resultados modelo 1")
start_time_1 = time.time()
docs_1 = vectorstore_1.similarity_search(query, k=3)
end_time_1 = time.time()
for i, doc in enumerate(docs_1):
    print(f"Chunk {i}:", doc.page_content+"\n")
print(f"Time taken (model 1): {end_time_1 - start_time_1:.4f} seconds")


# Resultados modelo 2
print("\nResultados modelo 2")
start_time_2 = time.time()
docs_2 = vectorstore_2.similarity_search(query, k=3)
end_time_2 = time.time()
for i, doc in enumerate(docs_2):
    print(f"Chunk {i}:", doc.page_content+"\n")
print(f"Time taken (model 2): {end_time_2 - start_time_2:.4f} seconds")

Resultados modelo 1
Chunk 0: that can be described as simple programs. He predicts that a realization of this within scientific communities will have a revolutionary influence on physics, chemistry, biology, and most other scientific areas, hence the book's title. The book was met with skepticism and criticism that Wolfram took credit for the work of others and made conclusions without evidence to support them.

Chunk 1: until Wolfram could publish the work in his controversial book A New Kind of Science. Wolfram's cellular-automata work came to be cited in more than 10,000 papers.

Chunk 2: Stephen Wolfram (2020). A Project to Find the Fundamental Theory of Physics. Champaign, Illinois: Wolfram Media. ISBN 978-1-57955-035-6.
Stephen Wolfram (2019). Adventures of a Computational Explorer. Champaign, Illinois: Wolfram Media. ISBN 978-1-57955-026-4.
Stephen Wolfram (2016). Idea Makers: Personal Perspectives on the Lives & Ideas of Some Notable People. Champaign, Illinois: Wolfram Media. 

## Parte 3: Generación de Respuestas
Aquí se evaluará la capacidad de algunos modelos para generar respuestas precisas a partir de los documentos recuperados

In [8]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 144 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (738 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [9]:
!rm -f ollama.tgz install.sh
!curl -fsSL https://ollama.com/install.sh -o install.sh
!sh install.sh
!which ollama
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


In [10]:
# Iniciar el servidor de Ollama si todavía no está activo
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

response = requests.get("http://127.0.0.1:11434/api/tags")
print(response.status_code)
print("Servidor de Ollama iniciado.")


200
Servidor de Ollama iniciado.


In [11]:
# Descargar el modelo. La descarga se realiza solo la primera vez.
import subprocess

subprocess.run(
    ["ollama", "pull", "phi3.5:latest"],
    check=True,
)

print("Modelo descargado correctamente.")


Modelo descargado correctamente.


In [12]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="phi3.5:latest",
    temperature=0.3,
)

print("Modelo cargado correctamente.")


Modelo cargado correctamente.


In [13]:
# Configuración de un pipeline RAG sencillo y compatible con LangChain actual
prompt = ChatPromptTemplate.from_template(
    """You are a question-answering assistant.

Use only the supplied context.
If the context does not contain the answer, say:
"I cannot answer that from the provided documents."

Context:
{context}

Question:
{question}

Answer briefly:"""
)

retriever = vectorstore_2.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)


def consultar_rag(pregunta, modelo=llm):
    documentos_recuperados = retriever.invoke(pregunta)
    contexto = "\n\n".join(
        f"[Source: {doc.metadata.get('title', 'Unknown')}]\n{doc.page_content}"
        for doc in documentos_recuperados
    )

    mensaje = prompt.format_messages(
        context=contexto,
        question=pregunta,
    )

    respuesta = modelo.invoke(mensaje)
    return {
        "answer": respuesta,
        "context": documentos_recuperados,
    }


question = "What software did Stephen Wolfram develop?"
result = consultar_rag(question)

print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.metadata.get('title')}: {doc.page_content[:150]}...")


Respuesta: Mathematica, a computer algebra system; Wolfram Alpha answer engine.

Chunks recuperados:

- Stephen Wolfram: Stephen Wolfram ( WUUL-frəm; born 29 August 1959) is a British-American computer scientist, physicist, and businessman. He works in computer algebra a...

- Stephen Wolfram: === Mathematica ===

In 1986, Wolfram left the Institute for Advanced Study for the University of Illinois Urbana–Champaign, where he had founded thei...

- Stephen Wolfram: === Symbolic Manipulation Program ===

Wolfram led the development of the computer algebra system SMP (Symbolic Manipulation Program) in the Caltech p...


In [14]:
temperatures = [0, 1]
out_of_context_question = "Should I drop university?"

for temperature in temperatures:
    print(f"\n--- Running with temperature={temperature} ---")

    current_llm = OllamaLLM(
        model="phi3.5:latest",
        temperature=temperature,
    )

    in_context_result = consultar_rag(question, modelo=current_llm)
    print("Respuesta a pregunta en contexto:")
    print(in_context_result["answer"])

    out_context_result = consultar_rag(
        out_of_context_question,
        modelo=current_llm,
    )
    print("\nRespuesta a pregunta fuera de contexto:")
    print(out_context_result["answer"])



--- Running with temperature=0 ---
Respuesta a pregunta en contexto:
Mathematica and the Symbolic Manipulation Program (SMP)

Respuesta a pregunta fuera de contexto:
The context does not provide specific advice on whether one should drop university; therefore, "I cannot answer that from the provided documents."

--- Running with temperature=1 ---
Respuesta a pregunta en contexto:
Mathematica and the Symbolic Manipulation Program (SMP)

Respuesta a pregunta fuera de contexto:
The context does not provide specific advice on whether one should drop university; it only details Stephen Wolfram's educational background and his pursuit of advanced studies in physics after leaving St. John's College, Oxford early without a degree at that time. Therefore, I cannot answer your question based on the provided documents.


In [15]:
# Casos de prueba
test_questions = [
    "What is the secret color of the universe?",
    "Las focas han migrado recientemente?",
    "Can you help me with my thesis? The deadline is tomorrow.",
]

for indice, test_question in enumerate(test_questions, start=1):
    result = consultar_rag(test_question)
    print(f"\nRespuesta {indice}:")
    print(result["answer"])



Respuesta 1:
I cannot answer that from the provided documents.

Respuesta 2:
I cannot answer that from the provided documents.

Respuesta 3:
I cannot answer that from the provided documents. For assistance with your thesis, especially under a tight deadline, it would be best to consult directly with your advisor or academic support services at your institution. They can provide personalized guidance and help you manage your time effectively.


In [16]:
# Variante de prompt más estricta para evitar respuestas no sustentadas
strict_prompt = ChatPromptTemplate.from_template(
    """You are a grounded RAG assistant.

Rules:
1. Answer only with information explicitly present in the context.
2. Do not use outside knowledge.
3. If the answer is absent, respond exactly:
   "I cannot answer that from the provided documents."

Context:
{context}

Question:
{question}

Answer:"""
)


def consultar_rag_estricto(pregunta, modelo=llm):
    documentos_recuperados = retriever.invoke(pregunta)
    contexto = "\n\n".join(
        f"[Source: {doc.metadata.get('title', 'Unknown')}]\n{doc.page_content}"
        for doc in documentos_recuperados
    )

    mensaje = strict_prompt.format_messages(
        context=contexto,
        question=pregunta,
    )

    respuesta = modelo.invoke(mensaje)
    return {
        "answer": respuesta,
        "context": documentos_recuperados,
    }

print("Pipeline RAG estricto configurado.")


Pipeline RAG estricto configurado.


In [17]:
# Casos de prueba con el prompt estricto
test_questions = [
    "What is the secret color of the universe?",
    "Las focas han migrado recientemente?",
    "Can you help me with my thesis? The deadline is tomorrow.",
]

for indice, test_question in enumerate(test_questions, start=1):
    result = consultar_rag_estricto(test_question)
    print(f"\nRespuesta {indice}:")
    print(result["answer"])



Respuesta 1:
I cannot answer that from the provided documents.

Respuesta 2:
I cannot answer that from the provided documents.

Respuesta 3:
I cannot answer that from the provided documents.
